In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel('FSO_feature.xlsx')

In [3]:
df 


,Time,ParticulateMax,TemperatureDifference,RelativeHumidity,VisibilityMax,TemperatureMax,TemperatureMin,ParticulateMin,VisibilityMin,AbsoluteHumidity,Particulate,Temperature,Visibility,Distance,FSO_Att
0,10,0.000000,0.000,80.291130,52745.193251,23.615676,21.173420,0.000000,47764.908850,15.930662,0.000000,22.384038,50251.391340,2959.931772,8.480261
1,11,0.000000,3.630,36.754752,19854.670076,16.687616,15.251449,0.000000,18626.944712,4.884168,0.000000,15.593858,19554.715500,2018.767588,3.511600
2,22,0.000000,0.250,81.129278,16669.585430,28.518049,26.583940,0.000000,14027.718822,21.586978,0.000000,27.592859,15287.325520,2956.858380,6.058910
3,12,0.000000,0.225,70.212589,25252.272628,2.824572,2.434717,0.000000,24293.881091,4.074320,0.000000,2.632763,25159.573780,4818.176068,6.593463
4,3,0.000000,0.020,68.977211,33269.672010,29.207603,24.685737,0.000000,28258.151377,17.840132,0.000000,27.080075,30345.736030,2955.199128,6.801351
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42859,5,18.215495,-0.200,76.290310,29515.848890,5.817576,5.462535,15.876605,27059.715033,5.437567,16.957986,5.722957,27589.122520,4825.841905,5.450631
42860,22,28.111408,-0.320,95.759465,13280.793041,23.635898,20.526191,25.549803,12142.272075,18.646348,27.278849,22.057799,12247.857120,2115.649872,8.562129
42861,0,188.515642,0.010,88.914893,8428.066276,15.285942,13.906865,172.608997,7303.756801,11.384310,174.912616,14.982220,7879.930898,2018.153150,5.831768
42862,15,5.964829,-0.400,76.946078,77990.816742,7.448264,6.373556,5.790368,72244.986530,5.938114,5.954184,6.939353,74106.624310,4819.435242,4.798019


In [4]:
X = df.drop(columns=["FSO_Att"])
y = df["FSO_Att"]

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, None],
    'min_samples_split': [2],
    'max_features': ['sqrt']
}


grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, None], 'max_features': ['sqrt'],
                         'min_samples_split': [2], 'n_estimators': [50, 100]},
             scoring='neg_root_mean_squared_error', verbose=1)

In [7]:
print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}


In [8]:
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

RandomForestRegressor(max_features='sqrt', random_state=42)

In [9]:
y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)

In [10]:
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))

In [11]:
r2_train = r2_score(y_train, y_train_pred)
r2_val = r2_score(y_val, y_val_pred)

In [12]:
print(f"Training RMSE: {rmse_train:.4f}, Validation RMSE: {rmse_val:.4f}")
print(f"Training R²: {r2_train:.4f}, Validation R²: {r2_val:.4f}")

Training RMSE: 0.3864, Validation RMSE: 1.0503
Training R²: 0.9901, Validation R²: 0.9299


## Adding predicted FSO

In [14]:


train_results = pd.DataFrame({
    "Original_FSO_Att": y_train.values,
    "Predicted_FSO_Att": y_train_pred
})
val_results = pd.DataFrame({
    "Original_FSO_Att": y_val.values,
    "Predicted_FSO_Att": y_val_pred
})

# Combine predictions
all_results = pd.concat([train_results, val_results])

# Merge predictions back on matching actual RFL_Att (retain original untouched)

train_original = pd.read_excel('train_dataset.xlsx')


In [15]:
train_original["Predicted_FSO_Att"] = None

for i in train_original.index:
    match = all_results[all_results["Original_FSO_Att"] == train_original.loc[i, "FSO_Att"]]
    if not match.empty:
        train_original.loc[i, "Predicted_FSO_Att"] = match.iloc[0]["Predicted_FSO_Att"]

In [16]:
train_original.to_excel("FSO_pred_dataset.xlsx", index=False)

## Test the Model

In [18]:
test_df = pd.read_excel('test_dataset.xlsx')

In [19]:
selected_features = [
    'Time', 'ParticulateMax', 'TemperatureDifference', 'RelativeHumidity', 'VisibilityMax',
    'TemperatureMax', 'TemperatureMin', 'ParticulateMin', 'VisibilityMin',
    'AbsoluteHumidity', 'Particulate', 'Temperature','Visibility', 'Distance', 'FSO_Att'
]

# Filter only the selected features (if they exist in the dataset)
filtered_df_fso = test_df[[col for col in selected_features if col in test_df.columns]]

In [20]:
X_test = filtered_df_fso.drop(columns=["FSO_Att"])
y_test = filtered_df_fso["FSO_Att"]

In [21]:
y_test_pred = best_model.predict(X_test)

In [22]:
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

In [23]:
print(f"Testing RMSE: {rmse_test:.4f}")
print(f"Testing R²: {r2_test:.4f}")

Testing RMSE: 1.0239
Testing R²: 0.9309


In [24]:
test_df["Predicted_FSO_Att"] = y_test_pred

# Save to a new Excel file
test_df.to_excel("test_FSO_dataset_with_predictions.xlsx", index=False)